In [ ]:
!pip install deep_translator

In [ ]:
# ============================================================
# ІМПОРТИ ТА НАЛАШТУВАННЯ
# ============================================================
import pandas as pd
import time
import re
from deep_translator import GoogleTranslator
translator = GoogleTranslator(source='en', target='uk')

In [ ]:
df = pd.read_csv('../data/raw/footnotes.csv')

In [ ]:
df_persons = pd.read_csv('../data/raw/persons_of_concern.csv')

In [ ]:
# ============================================================
# ФУНКЦІЇ ОЧИЩЕННЯ Й ПЕРЕКЛАДУ
# ============================================================

# Функція для очищення тексту перед перекладом
def clean_text_for_translator(text):
    if pd.isnull(text):
        return ""
    text_str = str(text)

    # Замінюємо довгі/фігурні тире (—, –, ‐) на звичайний дефіс (-)
    text_str = re.sub(r'[\u2010\u2011\u2012\u2013\u2014\u2015]', '-', text_str)

    # Замінюємо стрілочки "-->" або "->" на слово "to"
    text_str = text_str.replace("-->", " to ").replace("->", " to ")

    # Прибираємо зайві специфічні символи, залишаючи текст чистим
    return text_str.strip()

# Функція перекладу
def robust_translate(text):
    if pd.isnull(text) or str(text).strip() == "":
        return text

    # Очищуємо текст від проблемних дефісів та знаків
    cleaned_text = clean_text_for_translator(text)

    for attempt in range(3):
        try:
            time.sleep(0.3)  # Пауза для захисту від блокування
            return translator.translate(cleaned_text)
        except Exception:
            time.sleep(1.5)  # Якщо помилка, чекаємо довше і пробуємо знову
            continue

    # Для занадто довгих приміток перекладаємо, розбивши на речення
    try:
        parts = cleaned_text.split(". ")
        translated_parts = [translator.translate(p) for p in parts if p.strip() != ""]
        return ". ".join(translated_parts)
    except Exception:
        return text  # Якщо вже нічого не допомогло, повертаємо оригінал


In [ ]:
# ============================================================
# ПЕРЕКЛАД ПРИМІТОК (df, колонка Footnote)
# ============================================================
column_to_translate = 'Footnote'
df['Footnote_ukr'] = df[column_to_translate].apply(robust_translate)


In [ ]:
# ============================================================
# ПЕРЕКЛАД НАЗВ КРАЇН (df_persons, колонка Country_of_Asylum)
# ============================================================
unique_countries = df_persons['Country_of_Asylum'].dropna().unique()

country_map = {country: robust_translate(country) for country in unique_countries}
df_persons['Country_of_Asylum_ukr'] = df_persons['Country_of_Asylum'].map(country_map)


In [ ]:
# ============================================================
# ЗАПИС РЕЗУЛЬТАТІВ У ФАЙЛИ
# ============================================================
df.to_csv('../data/translated/footnotes_perfect_ukr.csv', index=False, encoding='utf-8-sig')
df_persons.to_csv('../data/translated/persons_of_concern_ukr.csv', index=False, encoding='utf-8-sig')
